# 1.4 dalis — CNN (konvoliucinis neuroninis tinklas)
### Kliento užklausos klasifikavimas pagal užklausos **vaizdą**

Originalus „Colab CNN“ moko konvoliucinį tinklą atpažinti vaizdų klases
(MNIST tipo duomenims: konvoliucija išskiria formas, telkimas (angl. *pooling*) mažina matmenis,
tankieji sluoksniai klasifikuoja). Jis buvo **suprastas ir perdarytas** taip:

**Mūsų adaptacija:** kliento užklausa **nupiešiama kaip pustonių (angl. *grayscale*) vaizdas**
(`PIL` biblioteka), o CNN mokosi iš šio vaizdo atpažinti ketinimo klasę.
Taip naudojame savo pačių (X, y) poras: X čia yra užklausos tekstas,
paverstas vaizdu, o ne žodžių eilutė — modelis turi atpažinti tekstą
vizualiai, taip pat kaip originalioje „Colab“ užrašinėje atpažįstami skaitmenys.

**Procesų seka (angl. *pipeline*):**
`Užklausa (X) → PIL teksto piešimas → 64×256 pustonių vaizdas → CNN → ketinimo klasė (y)`

> Patarimas: *Runtime → Change runtime type →* **T4 GPU** pagreitins mokymąsi,
> bet CPU taip pat pakanka (nedidelis duomenų kiekis).


## 1. Aplinkos paruošimas

In [ ]:
%pip install -q tensorflow Pillow scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

import tensorflow as tf
from PIL import Image, ImageDraw, ImageFont
import textwrap
tf.random.set_seed(SEED)
print('TensorFlow version:', tf.__version__)

## 2. Duomenys: (X, y) poros ir jų piešimas į vaizdus
`render_query()` nupiešia užklausą dviem eilutėmis 64×256 paveikslėlyje:
fonas — juodas, tekstas — baltas (kaip MNIST). Atsitiktinis poslinkis (angl. *jitter*) su
fiksuota pradine reikšme (angl. *seed*) sukuria nedidelę įvairovę, kad modelis neprisirištų
prie pikselių lygmeniu identiškų paveikslėlių.

In [ ]:
# --- Deterministic dataset generator (same as scripts/generate_dataset.py) ---
# X = customer query text, y = intent label (6 classes).
DATA_SPEC = {
    'order_status': [
        ('Kur mano užsakymas {order}?', 'order'),
        ('Kokia yra užsakymo {order} būsena?', 'order'),
        ('Nematau užsakymo {order} paskyroje, ką daryti?', 'order'),
        ('Kaip sužinoti mano užsakymo {order} statusą?', 'order'),
        ('Ar užsakymas {order} jau išsiųstas?', 'order'),
        ('Mano užsakymo {order} statusas vis dar „Apmokėta“, ar tai normalu?', 'order'),
        ('Kada bus matoma užsakymo {order} sekimo informacija?', 'order'),
    ],
    'delivery': [
        ('Kiek trunka pristatymas į {city}?', 'city'),
        ('Ar pristatote siuntas į {city} savaitgaliais?', 'city'),
        ('Kokia pristatymo į {city} kaina?', 'city'),
        ('Kada mano siunta pasieks {city}?', 'city'),
        ('Ar taikomas nemokamas pristatymas perkant už {eur} eurų?', 'eur'),
        ('Kur rasti siuntos į {city} sekimo numerį?', 'city'),
        ('Ar galiu atsiimti siuntą į {city} paštomate?', 'city'),
    ],
    'returns': [
        ('Noriu grąžinti {item}, kaip tai padaryti?', 'item'),
        ('Ar galiu grąžinti prekę po {days} dienų?', 'days'),
        ('Kiek kainuoja grąžinti {item}?', 'item'),
        ('Kada atgausiu pinigus grąžinęs {item}?', 'item'),
        ('Ar galima grąžinti {item} originalioje pakuotėje?', 'item'),
        ('Kur rasiu grąžinimo formą norėdamas grąžinti {item}?', 'item'),
        ('Ar galima grąžinti {item}, jei pakuotė jau buvo atidaryta?', 'item'),
    ],
    'payment': [
        ('Ar priimate {method}?', 'method'),
        ('Kiek laiko trunka mokėjimo autorizacija pasirinkus {method}?', 'method'),
        ('Nepavyko apmokėti užsakymo pasirinkus {method}, ką daryti?', 'method'),
        ('Ar galima mokėti dalimis pasirinkus {method}?', 'method'),
        ('Kada gausiu sąskaitą faktūrą apmokėjus per {method}?', 'method'),
        ('Ar palaikote {method} internetinėje parduotuvėje?', 'method'),
        ('Ką daryti, jei mokant per {method} nurašė pinigus, bet užsakymas neapmokėtas?', 'method'),
    ],
    'warranty': [
        ('Kiek laiko galioja garantija įsigijus {item}?', 'item'),
        ('Ar nemokamai suremontuosite {item} pagal garantiją?', 'item'),
        ('Ką daryti, jei gaminys sugedo po {days} dienų naudojimo?', 'days'),
        ('Ar reikia atskiro talono pateikiant {item} garantiniam remontui?', 'item'),
        ('Kiek trunka garantinis remontas pateikus {item}?', 'item'),
        ('Ar garantija galioja, jei netyčia pažeidžiau {item}?', 'item'),
        ('Ar atvežate {item} į remontą nemokamai?', 'item'),
    ],
    'product_usage': [
        ('Kaip susieti {item} su telefonu?', 'item'),
        ('Kur rasti naudotojo vadovą įsigijus {item}?', 'item'),
        ('Nusipirkau {item}, bet prietaisas neįsijungia, ką daryti?', 'item'),
        ('Kaip atstatyti gamyklinius nustatymus naudojant {item}?', 'item'),
        ('Ar galima gauti naudotojo vadovą lietuvių kalba įsigijus {item}?', 'item'),
        ('Kaip tinkamai prijungti {item} prie kompiuterio?', 'item'),
        ('Kiek laiko reikia krauti {item}?', 'item'),
    ],
}

SLOT_VALUES = {
    'order': ['TS-102345', 'TS-201111', 'TS-314560', 'TS-456789', 'TS-511235', 'TS-623456'],
    'city': ['Vilnių', 'Kauną', 'Klaipėdą', 'Šiaulius', 'Panevėžį', 'Alytų'],
    'item': ['išmanųjį telefoną', 'belaides ausines', 'išorinę bateriją',
             'elektroninę skaityklę', 'žaidimų pultelį', 'internetinę kamerą'],
    'days': ['3', '7', '10', '13', '20', '30'],
    'eur': ['30', '45', '60', '75', '90', '120'],
    'method': ['PayPal', 'Visa kortelę', 'Mastercard kortelę',
               'bankinį pavedimą', 'Google Pay', 'Apple Pay'],
}

INTENTS = ['order_status', 'delivery', 'returns', 'payment', 'warranty', 'product_usage']


def generate_dataset():
    rows = []
    for intent, specs in DATA_SPEC.items():
        for template, slot in specs:
            for value in SLOT_VALUES[slot]:
                rows.append((template.replace('{' + slot + '}', value), intent))
    return pd.DataFrame(rows, columns=['query', 'intent'])


df = generate_dataset()
print(f'Dataset: {len(df)} (X, y) pairs, {len(INTENTS)} classes')
print(df['intent'].value_counts())

# Canonical hold-out test set (same 6 queries as data/customer_support_test.csv
# and as the 1.2.2 Gemini prototype) - used to compare all models fairly.
TEST_EXAMPLES = [
    ('Kokiomis dienomis pristatote užsakymus į Vilnių?', 'delivery'),
    ('Norėčiau grąžinti prekę po 10 dienų nuo pirkimo, ar įmanoma?', 'returns'),
    ('Nematau savo užsakymo istorijos paskyroje, ką daryti?', 'order_status'),
    ('Ar parduotuvė priima PayPal mokėjimus?', 'payment'),
    ('Ar už 25 eurus pirktas gaminys turi garantiją?', 'warranty'),
    ('Kur rasti ausinių susiejimo instrukciją?', 'product_usage'),
]

IMG_W, IMG_H = 256, 64

def get_font(size=22):
    for path in ['/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
                 'C:/Windows/Fonts/arial.ttf']:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            continue
    return ImageFont.load_default()

def render_query(text, idx, img_w=IMG_W, img_h=IMG_H):
    """Draw a query string as a binary (0/1) grayscale image with slight jitter."""
    rng = np.random.RandomState(idx)  # deterministic jitter per example
    img = Image.new('L', (img_w, img_h), color=0)
    draw = ImageDraw.Draw(img)
    wrapped = textwrap.wrap(text, width=28)[:2]
    y = 8 + rng.randint(-2, 3)
    for line in wrapped:
        draw.text((8 + rng.randint(-2, 3), y), line, fill=255, font=get_font(20))
        y += 24
    arr = np.asarray(img, dtype='float32') / 255.0
    return arr

print('Sample rendering:')
plt.imshow(render_query(TEST_EXAMPLES[0][0], 0), cmap='gray')
plt.axis('off'); plt.title(TEST_EXAMPLES[0][0]); plt.show()

In [ ]:
import textwrap

# Render the whole dataset
X_images = np.stack([render_query(q, i) for i, q in enumerate(df['query'])])
y_labels = np.array([INTENTS.index(l) for l in df['intent']])
print('Image dataset:', X_images.shape)

# Show one rendered image per intent class
fig, axes = plt.subplots(2, 3, figsize=(12, 4))
for k, ax in enumerate(axes.flat):
    idx = int(np.where(y_labels == k)[0][0])
    ax.imshow(X_images[idx], cmap='gray')
    ax.set_title(f'{INTENTS[k]}: {df["query"].iloc[idx][:35]}...')
    ax.axis('off')
plt.suptitle('Užklausų vaizdai pagal klasę')
plt.tight_layout(); plt.show()

## 3. CNN architektūra

| Sluoksnis | Paskirtis |
|---|---|
| `Conv2D(32, 3×3, relu)` | 32 formų filtrai (kraštinės, brūkšniai) |
| `MaxPooling(2×2)` | agreguoja 2×2 sritį → mažesnė raiška, invariantiškumas poslinkiui |
| `Conv2D(64, 3×3, relu)` | sudėtingesni formų deriniai |
| `MaxPooling(2×2)` | tolesnis matmenų sumažinimas |
| `Flatten` + `Dense(128)` | klasifikavimo galvutė |
| `Dropout(0.4)` | reguliarizacija |
| `Dense(6, softmax)` | tikimybės 6 klasėms |

Tai standartinis „konvoliucija × 2 → tankieji sluoksniai“ modelis, kaip ir
originalioje „Colab“ užrašinėje (ten tokiu pačiu principu klasifikuojami MNIST skaitmenys).


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_images, df['intent'], test_size=0.2, random_state=SEED, stratify=df['intent'])

X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
y_train = np.array([INTENTS.index(l) for l in y_train_raw])
y_val = np.array([INTENTS.index(l) for l in y_val_raw])

X_test = np.stack([render_query(q, 10_000 + i) for i, (q, _) in enumerate(TEST_EXAMPLES)])[..., np.newaxis]
y_test = np.array([INTENTS.index(l) for _, l in TEST_EXAMPLES])
print(f'Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}')

cnn_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(IMG_H, IMG_W, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(INTENTS), activation='softmax'),
])
cnn_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
cnn_model.summary()

## 4. Mokymasis ir kreivės

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy', patience=5, restore_best_weights=True)

cnn_history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=25, batch_size=16,
    callbacks=[early_stop], verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(cnn_history.history['loss'], label='mokymo nuostoliai')
axes[0].plot(cnn_history.history['val_loss'], label='validacijos nuostoliai')
axes[0].set_xlabel('Epocha'); axes[0].set_ylabel('Nuostoliai (Loss)'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(cnn_history.history['accuracy'], label='mokymo tikslumas')
axes[1].plot(cnn_history.history['val_accuracy'], label='validacijos tikslumas')
axes[1].set_xlabel('Epocha'); axes[1].set_ylabel('Tikslumas (Accuracy)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('CNN mokymosi kreivės')
plt.tight_layout(); plt.show()

## 5. Vertinimas

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Numatytas ketinimas')
    ax.set_ylabel('Tikrasis ketinimas')
    ax.set_title(title)
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
    return cm

from sklearn.metrics import accuracy_score, classification_report

val_pred = np.argmax(cnn_model.predict(X_val, verbose=0), axis=1)
test_pred = np.argmax(cnn_model.predict(X_test, verbose=0), axis=1)

print(f'Validacijos tikslumas: {accuracy_score(y_val, val_pred):.4f}')
print(f'Testavimo (6 užklausos) tikslumas: {accuracy_score(y_test, test_pred):.4f}')
print()
for (query, true_label), pred_id in zip(TEST_EXAMPLES, test_pred):
    mark = 'OK ' if INTENTS[pred_id] == true_label else 'WRONG'
    print(f'{mark} true={true_label:14s} pred={INTENTS[pred_id]:14s} {query}')

plot_confusion_matrix(y_val, val_pred, INTENTS, 'CNN — validacijos painiavos matrica')
print(classification_report(y_val, val_pred, target_names=INTENTS))

## 6. Išvados
1. CNN sėkmingai klasifikuoja užklausos **vaizdus** — teksto piešinys atlieka
   MNIST vaidmenį originalioje „Colab“ užrašinėje, o klasės yra ketinimai.
2. CNN mokosi vizualinių formų (raidžių kraštinių), o ne žodžių reikšmių — todėl
   šis modelis yra mažiau tinkamas metodas tekstui klasifikuoti nei 1.3(a) MLP su
   TF-IDF požymiais, tačiau puikiai demonstruoja konvoliucijos principą.
3. Svarbiausia įžvalga: tos pačios (X, y) poros gali būti pavaizduotos ir
   kaip tekstas, ir kaip vaizdas — modelio pasirinkimas priklauso nuo duomenų
   reprezentacijos.
